In [ ]:
%pip install -q otter-grader

# In-Class Exercise 05: Probabilistic Modeling — fit, check, simulate

**DS701 — Session 6 (Wed Sep 23, 2026) — Section A1**

[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tools4ds/DS701-Materials-FA26/blob/main/class_activity_notebooks/05-InClass-Exercise-Probabilistic-Modeling-A1/05-InClass-Exercise-Probabilistic-Modeling-A1.ipynb)

**Time: ~60 minutes.** Work in groups of 2–3.

Parts marked **(autograded)** are submitted to Gradescope; open-ended parts are graded
for participation. You may use AI assistance, but you must be able to explain and
justify every part of your solution when asked — staff will circulate and ask.

**Plan**

| Part | Topic | Grading | Time |
|---|---|---|---|
| 1 | Fit a Gaussian to a month of Boston temperatures — by hand and with `scipy` | autograded | ~12 min |
| 2 | Check the fit: $\pm k\sigma$ fractions and a tail probability | autograded | ~10 min |
| 3 | Fit a Poisson to the horse-kick counts and check it | autograded | ~10 min |
| 4 | **Stretch** — simulate: the Law of Large Numbers | autograded | ~8 min |
| 5 | **Stretch** — simulate: the Central Limit Theorem | autograded | ~8 min |
| 6 | **Stretch** — what "95%" means: confidence-interval coverage | autograded + written | ~12 min |

Dependencies: `numpy`, `pandas`, `matplotlib`, `scipy`.

## Setup

The lecture's recipe, which we follow twice and then put to the test:

1. **look** at the data and pick a family of distributions whose shape matches,
2. **estimate** its parameters (by maximum likelihood),
3. **check** the fit,
4. **use** the model.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm, poisson

In [ ]:
# Lecture section. Only A1 runs this semester; this is kept because the build
# script reads it to name the variant directory the notebook is published in.
SECTION = "A1"
SEED = 701

In [ ]:
# Colab: fetch the autograder tests for this activity.
import os, sys, urllib.request

if "google.colab" in sys.modules:
    os.makedirs("tests", exist_ok=True)
    BASE = "https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main/class_activity_notebooks/05-InClass-Exercise-Probabilistic-Modeling-A1/tests/"
    for _q in ("q1", "q2", "q3", "q4", "q5", "q6"):
        urllib.request.urlretrieve(BASE + _q + ".py", "tests/" + _q + ".py")

In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook()

### Data 1: Boston daily mean temperatures

Same source as the lecture — NOAA daily records for Boston Logan (station
`USW00014739`), with `TMEAN = (TMAX + TMIN) / 2` in °C. To keep the classroom
independent of the NOAA server we read a CSV snapshot that was downloaded once with
the lecture's helper. Only complete years (1936–2025) are used, so everyone in your
section sees exactly the same numbers.

We work with **August**.

In [ ]:
def load_boston_temps():
    """DataFrame with DATE, TMAX, TMIN, TMEAN (°C), Boston Logan, 1936-."""
    # 1) local checkout of the course repo (staff / offline)
    for p in ["data/boston_daily_temps.csv", "../data/boston_daily_temps.csv",
              "../../data/boston_daily_temps.csv"]:
        if os.path.exists(p):
            return pd.read_csv(p)
    # 2) the snapshot served by the course website
    try:
        return pd.read_csv(
            "https://tools4ds.github.io/ds701/data/boston_daily_temps.csv")
    except Exception as e:
        print("course-site copy unavailable:", e, "-- falling back to NOAA")
    # 3) last resort: NOAA directly (the lecture's helper)
    import requests
    params = {"dataset": "daily-summaries", "stations": "USW00014739",
              "startDate": "1936-01-01", "endDate": "2025-12-31",
              "dataTypes": "TMAX,TMIN", "format": "json", "units": "metric"}
    r = requests.get("https://www.ncei.noaa.gov/access/services/data/v1",
                     params=params, timeout=60)
    r.raise_for_status()
    df = pd.DataFrame(r.json())
    for c in ["TMAX", "TMIN"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=["TMAX", "TMIN"])
    df["TMEAN"] = (df["TMAX"] + df["TMIN"]) / 2
    return df


temps = load_boston_temps()
temps["DATE"] = pd.to_datetime(temps["DATE"])
temps = temps[temps["DATE"].dt.year <= 2025]          # complete years only
temps["MONTH"] = temps["DATE"].dt.month
temps["YEAR"] = temps["DATE"].dt.year

MONTH = 8
MONTH_NAME = {8: "August", 6: "June"}[MONTH]

x = temps.loc[temps["MONTH"] == MONTH, "TMEAN"].to_numpy()
N = len(x)
print(f"{len(temps):,} daily records total; {N:,} {MONTH_NAME} days "
      f"({temps['YEAR'].min()}-{temps['YEAR'].max()})")

plt.figure(figsize=(7, 4))
plt.hist(x, bins=40, density=True, alpha=0.6, edgecolor="black")
plt.title(f"Boston daily mean temperature in {MONTH_NAME} ({N:,} days)")
plt.xlabel("Temperature (°C)"); plt.ylabel("Density")
plt.show()

### Data 2: deaths by horse kick

Bortkiewicz's counts of Prussian cavalry soldiers killed by horse kicks, per corps per
year, 1875–1894 (the same table as `data/HorseKicks.txt` in the lecture, embedded here
so nothing needs downloading).

- Following Bortkiewicz we **drop** the four atypical corps
  (`GC`, `C1`, `C6`, `C11`): 10 corps × 20 years = 200 corps-years.

In [ ]:
from io import StringIO

_HORSE_KICKS = """Year,GC,C1,C2,C3,C4,C5,C6,C7,C8,C9,C10,C11,C14,C15
1875,0,0,0,0,0,0,0,1,1,0,0,0,1,0
1876,2,0,0,0,1,0,0,0,0,0,0,0,1,1
1877,2,0,0,0,0,0,1,1,0,0,1,0,2,0
1878,1,2,2,1,1,0,0,0,0,0,1,0,1,0
1879,0,0,0,1,1,2,2,0,1,0,0,2,1,0
1880,0,3,2,1,1,1,0,0,0,2,1,4,3,0
1881,1,0,0,2,1,0,0,1,0,1,0,0,0,0
1882,1,2,0,0,0,0,1,0,1,1,2,1,4,1
1883,0,0,1,2,0,1,2,1,0,1,0,3,0,0
1884,3,0,1,0,0,0,0,1,0,0,2,0,1,1
1885,0,0,0,0,0,0,1,0,0,2,0,1,0,1
1886,2,1,0,0,1,1,1,0,0,1,0,1,3,0
1887,1,1,2,1,0,0,3,2,1,1,0,1,2,0
1888,0,1,1,0,0,1,1,0,0,0,0,1,1,0
1889,0,0,1,1,0,1,1,0,0,1,2,2,0,2
1890,1,2,0,2,0,1,1,2,0,2,1,1,2,2
1891,0,0,0,1,1,1,0,1,1,0,3,3,1,0
1892,1,3,2,0,1,1,3,0,1,1,0,1,1,0
1893,0,1,0,0,0,1,0,2,0,0,1,3,0,0
1894,1,0,0,0,0,0,0,0,1,0,1,1,0,0
"""
horse_kicks = pd.read_csv(StringIO(_HORSE_KICKS), index_col="Year")

DROP_CORPS = ["GC", "C1", "C6", "C11"]
counts = horse_kicks.drop(columns=DROP_CORPS).to_numpy().ravel().astype(int)
print(f"{len(counts)} corps-years, {counts.sum()} deaths in total")
horse_kicks.head()

## Part 1 (autograded): fit a Gaussian by maximum likelihood

The histogram is unimodal and roughly bell-shaped, so we model each day's mean
temperature as a draw from $\mathcal{N}(\mu, \sigma^2)$. The lecture showed that
maximizing the log-likelihood

$$
\ell(\mu, \sigma^2) = -\frac{N}{2}\log(2\pi\sigma^2) - \frac{1}{2\sigma^2}\sum_{n=1}^{N}(x_n-\mu)^2
$$

gives the closed-form maximum likelihood estimates

$$
\hat\mu = \frac{1}{N}\sum_{n=1}^{N} x_n, \qquad
\hat\sigma^2 = \frac{1}{N}\sum_{n=1}^{N}(x_n - \hat\mu)^2 .
$$

**Task.** Using only `numpy` arithmetic on the array `x` (sums, squares, `len`; no
`np.mean`/`np.std`/`np.var` and no `scipy` yet):

- `mu_hat` — the MLE of $\mu$,
- `sigma2_hat` — the MLE of $\sigma^2$ (note: divide by $N$, not $N-1$),
- `sigma_hat` — its square root.

Then check yourself against `scipy`: `norm.fit(x)` returns the MLE `(loc, scale)` —
store them as `mu_sp`, `sigma_sp`. They should agree with yours to floating-point
precision. All five should be plain Python `float`s.

In [ ]:
# MLE by hand: sums and squares only.
mu_hat = ...
sigma2_hat = ...
sigma_hat = ...

# The same fit from scipy (norm.fit returns the MLE (loc, scale)).
mu_sp, sigma_sp = ...

print(f"by hand:  mu = {mu_hat:.4f} °C   sigma^2 = {sigma2_hat:.4f}   sigma = {sigma_hat:.4f} °C")
print(f"scipy:    mu = {mu_sp:.4f} °C   sigma = {sigma_sp:.4f} °C")

Now **look**: overlay the fitted density on the histogram (given).

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(x, bins=40, density=True, alpha=0.6, edgecolor="black", label="observed")
xs = np.linspace(x.min() - 2, x.max() + 2, 300)
plt.plot(xs, norm.pdf(xs, mu_hat, sigma_hat), "r-", lw=3,
         label=f"fitted Gaussian\n$\\mu$={mu_hat:.1f}, $\\sigma$={sigma_hat:.1f}")
plt.title(f"Boston {MONTH_NAME}: data vs. fitted Gaussian")
plt.xlabel("Temperature (°C)"); plt.ylabel("Density"); plt.legend()
plt.show()

In [ ]:
grader.check("q1")

## Part 2 (autograded): check the fit, then use the model

A model we do not check is just an assumption. Two checks from the lecture, then one
use.

**Task.**

1. `frac_within_1`, `frac_within_2` — the fraction of days with
   $|x - \hat\mu| \le k\hat\sigma$ for $k = 1, 2$. A Gaussian predicts about
   0.683 and 0.954.
2. A **tail probability**, model vs. data. Let `T0` be the threshold below
   (28 °C). Compute
   - `p_tail_model` — $P(X > T_0)$ under your fitted Gaussian (`norm.sf` is the upper
     tail, $1 - F$), and
   - `p_tail_emp` — the fraction of observed days with `x > T0`.
3. `n_expected_hot` — the number of days out of the $N$ observed that the *model* says
   should exceed `T0` (a float; compare it with the observed count in the print-out).

Then discuss at your table: does the Gaussian over- or under-state the warm tail for
your month? What in the histogram explains that?

In [ ]:
T0 = 28.0

frac_within_1 = ...
frac_within_2 = ...

p_tail_model = ...
p_tail_emp = ...
n_expected_hot = ...

print(f"within 1 sigma: observed {frac_within_1:.3f}   Gaussian predicts {norm.cdf(1) - norm.cdf(-1):.3f}")
print(f"within 2 sigma: observed {frac_within_2:.3f}   Gaussian predicts {norm.cdf(2) - norm.cdf(-2):.3f}")
print(f"P(T > {T0:.0f} °C):  model {p_tail_model:.4f}   empirical {p_tail_emp:.4f}")
print(f"days above {T0:.0f} °C out of {N}:  model expects {n_expected_hot:.1f}, observed {int((x > T0).sum())}")

In [ ]:
grader.check("q2")

## Part 3 (autograded): fit a Poisson to counts and check it

Small non-negative counts call for a different family. The Poisson has one parameter,
$\lambda$, and (as the lecture derived) its MLE is the sample mean count.

**Task.**

1. `lam_hat` — the MLE of $\lambda$ from `counts` (a float).
2. `expected_counts` — a NumPy array of length 7: for $k = 0, \dots, 6$, the number of
   corps-years (out of `len(counts)`) the fitted Poisson predicts to have exactly $k$
   deaths (`poisson.pmf`).
3. `observed_counts` — the observed number of corps-years with $k$ deaths, $k = 0,\dots,6$
   (`np.bincount(..., minlength=7)` is handy).
4. The property check: `var_over_mean` — the sample variance (divide by $N$) divided by
   the sample mean. For a Poisson this should be near 1.
5. Use the model: `p_ge3_model` — $P(X \ge 3)$ under the fitted Poisson (careful with
   `sf`: `poisson.sf(k, lam)` is $P(X > k)$), and `p_ge3_emp` — the observed fraction of
   corps-years with 3 or more deaths.

In [ ]:
ks = np.arange(0, 7)

lam_hat = ...
expected_counts = ...
observed_counts = ...
var_over_mean = ...
p_ge3_model = ...
p_ge3_emp = ...

print(f"lambda_hat = {lam_hat:.3f} deaths per corps per year   (variance/mean = {var_over_mean:.3f})")
print(f"P(3 or more deaths in a corps-year):  model {p_ge3_model:.4f}   empirical {p_ge3_emp:.4f}")

fit_table = pd.DataFrame({"observed": observed_counts,
                          "Poisson predicts": expected_counts.round(2)},
                         index=pd.Index(ks, name="deaths per corps-year"))
display(fit_table)
fit_table.plot.bar(figsize=(7, 4))
plt.ylabel(f"corps-years (out of {len(counts)})"); plt.xticks(rotation=0)
plt.show()

In [ ]:
grader.check("q3")

# Stretch: simulation — what the theorems *look like*

The lecture stated two facts about sample means and used them to build a confidence
interval:

- **Law of Large Numbers (LLN)** — the sample mean of $n$ i.i.d. draws converges to
  the true mean as $n \to \infty$.
- **Central Limit Theorem (CLT)** — the sample mean is approximately Gaussian with
  standard deviation $\sigma/\sqrt{n}$ (the *standard error*), whatever the shape of
  the individual draws.

Now that you have a fitted Poisson, use it as a **known population**: its true mean is
$\lambda = $ `lam_hat`, its variance is also $\lambda$, and it is decidedly not
bell-shaped. Everything below uses a seeded generator so results are reproducible and
gradeable — **use the `rng` objects exactly as given.**

## Part 4 (autograded, stretch): the Law of Large Numbers, watched

**Task.** `draws` below is a single sequence of $M = 5000$ Poisson($\hat\lambda$)
draws. Compute

- `running_mean` — a NumPy array of length 5000 whose entry $m$ (0-based) is the mean of
  the first $m+1$ draws (`np.cumsum` does it in one line), and
- `err_at` — a dict with keys `10, 100, 1000, 5000` giving the absolute error
  $|\bar{x}_n - \lambda|$ of the running mean after that many draws.

Then plot the running mean against $n$ on a log axis (given) — the picture *is* the
LLN.

In [ ]:
M = 5000
rng = np.random.default_rng(SEED)
draws = rng.poisson(lam_hat, size=M)          # one long sequence from the fitted population

running_mean = ...
err_at = ...

for n, e in err_at.items():
    print(f"after {n:>5} draws: running mean = {running_mean[n - 1]:.4f}, |error| = {e:.4f}")

plt.figure(figsize=(7, 4))
plt.plot(np.arange(1, M + 1), running_mean, lw=1.5)
plt.axhline(lam_hat, color="r", ls="--", label=f"true mean $\\lambda$ = {lam_hat:.3f}")
plt.xscale("log"); plt.xlabel("number of draws n (log scale)"); plt.ylabel("running mean")
plt.title("Law of Large Numbers: the sample mean settles down"); plt.legend()
plt.show()

In [ ]:
grader.check("q4")

## Part 5 (autograded, stretch): the Central Limit Theorem, watched

Now many short experiments instead of one long one. For each sample size $n$ in
`n_values = [1, 2, 5, 30, 200]`, draw `R = 4000` independent samples of size $n$ from
Poisson($\hat\lambda$) and record the $R$ sample means.

**Task.**

1. Write `sample_means(n, R, rng)` returning a length-`R` array of sample means, each
   the mean of `n` draws. Use `rng.poisson(lam_hat, size=(R, n)).mean(axis=1)` — one
   call, so the seed sequence is the same for everyone.
2. Fill the dict `sd_of_means`: for each $n$ in `n_values`, the standard deviation of
   the $R$ sample means (a float).
3. Fill `sd_predicted`: the CLT's prediction $\sqrt{\lambda / n}$ for each $n$
   (recall a Poisson's variance equals its mean).

Then run the plotting cell and *look*: at which $n$ does the histogram of means start to
look Gaussian? Compare with the lecture's "$n \ge 30$" rule of thumb.

In [ ]:
n_values = [1, 2, 5, 30, 200]
R = 4000
rng = np.random.default_rng(SEED)   # fresh generator; use it in the order n_values lists n


def sample_means(n, R, rng):
    """R sample means, each of n Poisson(lam_hat) draws."""
    ...


means_by_n = {n: sample_means(n, R, rng) for n in n_values}
sd_of_means = ...
sd_predicted = ...

for n in n_values:
    print(f"n = {n:>3}:  sd of sample means = {sd_of_means[n]:.4f}   CLT predicts sqrt(lambda/n) = {sd_predicted[n]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, len(n_values), figsize=(16, 3.4))
for ax, n in zip(axes, n_values):
    m = means_by_n[n]
    bins = np.arange(m.min() - 0.5 / n, m.max() + 1 / n, 1 / n) if n <= 30 else 30
    ax.hist(m, bins=bins, density=True, alpha=0.6, edgecolor="black")
    xs = np.linspace(m.min(), m.max(), 200)
    ax.plot(xs, norm.pdf(xs, lam_hat, np.sqrt(lam_hat / n)), "r-", lw=2)
    ax.set_title(f"means of n = {n} draws")
plt.suptitle("CLT: distribution of the sample mean (bars) vs. the Gaussian it predicts (red)")
plt.tight_layout(); plt.show()

In [ ]:
grader.check("q5")

## Part 6 (autograded, stretch): what does "95%" mean?

The lecture's confidence interval for a mean:

$$
\bar{x} \;\pm\; z_{0.975}\,\frac{s}{\sqrt{n}}, \qquad z_{0.975} \approx 1.96,
\quad s = \sqrt{\tfrac{1}{n-1}\sum (x_i - \bar{x})^2}.
$$

Here you can do something you can never do with real data: you **know the true mean**
($\lambda = $ `lam_hat`), so you can check how often the interval actually contains it.

**Task.**

1. Write `ci95(sample)` returning a tuple `(lo, hi)` for a 1-D array `sample`, using
   `norm.ppf(0.975)` for $z$ and `ddof=1` for $s$.
2. Write `coverage_and_width(samples, true_mean)`: for a 2-D array whose rows are
   independent samples, return `(coverage, mean_width)` — the fraction of rows whose
   `ci95` contains `true_mean`, and the average interval width (both floats).
3. The given lines then apply it to `samples_30` (`R = 2000` samples of size $n = 30$),
   `samples_5` and `samples_120`, producing `coverage_30`, `coverage_5`,
   `mean_width_30`, `mean_width_120`, ... How does the width at $n = 120$ compare with
   $n = 30$?

The plotting cell draws the first 100 intervals for $n = 30$, colouring the misses.

In [ ]:
R = 2000
rng = np.random.default_rng(SEED + 1)
samples_30 = rng.poisson(lam_hat, size=(R, 30))
samples_5 = rng.poisson(lam_hat, size=(R, 5))
samples_120 = rng.poisson(lam_hat, size=(R, 120))


def ci95(sample):
    """95% confidence interval (lo, hi) for the mean of a 1-D sample."""
    ...


def coverage_and_width(samples, true_mean):
    """Fraction of rows whose CI contains true_mean, and the mean CI width."""
    ...


coverage_30, mean_width_30 = coverage_and_width(samples_30, lam_hat)
coverage_5, mean_width_5 = coverage_and_width(samples_5, lam_hat)
coverage_120, mean_width_120 = coverage_and_width(samples_120, lam_hat)

print(f"n = 30 : coverage {coverage_30:.3f}   mean width {mean_width_30:.3f}")
print(f"n = 5  : coverage {coverage_5:.3f}   mean width {mean_width_5:.3f}")
print(f"n = 120: coverage {coverage_120:.3f}   mean width {mean_width_120:.3f}   (width ratio 30/120 = {mean_width_30 / mean_width_120:.2f})")

In [ ]:
# The first 100 intervals for n = 30: blue if it covers lambda, red if it misses.
plt.figure(figsize=(7, 6))
for i, row in enumerate(samples_30[:100]):
    lo, hi = ci95(row)
    hit = lo <= lam_hat <= hi
    plt.plot([lo, hi], [i, i], color="tab:blue" if hit else "tab:red", lw=1.5)
plt.axvline(lam_hat, color="k", ls="--", label=f"true mean {lam_hat:.3f}")
plt.xlabel("interval for the mean"); plt.ylabel("replication")
plt.title(f"100 of the {R} intervals (n = 30) -- coverage over all {R}: {coverage_30:.1%}")
plt.legend(); plt.show()

In [ ]:
grader.check("q6")

<!-- BEGIN QUESTION -->

## Part 7 (written, participation): what a confidence interval does — and does not — mean

Fill in your numbers and answer in a few sentences each. **Be ready to explain your
reasoning when staff visit your table.**

1. *"My 95% intervals covered the true mean **___%** of the time for $n = 30$ and
   **___%** for $n = 5$."* Explain any gap from 95%. (Hint: which assumption of the
   CLT-based interval is shakiest for 5 draws from a Poisson with $\lambda < 1$? What
   happens to the interval when all 5 draws are 0?)
2. In Part 2 you fitted a Gaussian to a whole month of days. Suppose you reported a 95%
   confidence interval for the **mean** temperature. Say precisely what that interval
   does **not** tell you about *individual* days, and which two numbers from Part 1
   *would* answer that question.
3. A classmate says: "there is a 95% probability that the true mean lies in *my*
   interval [0.52, 0.71]." Rewrite the sentence so that it is correct, and say what is
   random and what is fixed.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

## Wrap-up

You just ran the lecture's whole pipeline twice — *look → estimate (MLE) → check → use*
— on a continuous variable (Gaussian) and on counts (Poisson), and then used a fitted
model as a known population to **see** the Law of Large Numbers, the Central Limit
Theorem, and what a 95% confidence interval promises (and what it does not).

Next lecture the same maximum-likelihood idea meets a harder problem: several Gaussians
mixed together, with unknown membership — Gaussian Mixture Models and EM.

Before you leave, make sure everyone at your table can answer, without notes: *why is
the MLE of a Gaussian mean the sample mean?* and *why did the $n = 5$ intervals
under-cover?*